# Step 4 — Build Feature Blocks @ baseline (t0)

## 🎯 Objetivo
Construir as *feature tables* (1 linha por doente) usando apenas informação disponível **até ao baseline (t0)**:
- **Vitals @ t0** (última medição pré-t0)
- **Respiratório (FVC) @ t0** (última medição pré-t0; melhor tentativa)
- **Tabela final**: baseline + vitals + FVC (merge por doente)
- **Coverage table** para documentar percentagens de disponibilidade por bloco

## 📥 Inputs (raw/interim)
- `01_data/interim/baseline_targets_slopes_v1.csv` *(ou o nome equivalente do Step 3A, 1 linha por doente, inclui `t0` e `subject_id`)*
- `01_data/raw/PROACT_VITALSIGNS.csv`
- `01_data/raw/PROACT_FVC.csv` *(e/ou SVC, se aplicável)*

## 📤 Outputs (interim/processed)
- `01_data/processed/features_vitals_t0.csv`
- `01_data/processed/features_fvc_t0.csv`
- `01_data/processed/features_baseline_v1.csv`
- `04_outputs/tables/step4_coverage_features.csv`

## ⚙️ Princípio temporal (leakage-free)
Todas as features são extraídas com a regra:
> **usar apenas registos com delta ≤ t0**, escolhendo a **última medição antes do baseline** (last pre-baseline).

<div style="padding:10px;border-left:6px solid #FF5F5D;">
<b>Nota:</b> este notebook não treina modelos. Apenas prepara features e documenta coverage para suportar decisões metodológicas no Step 5.
</div>


In [1]:
import os
import numpy as np
import pandas as pd

# Paths do projeto (ajusta se a tua raiz for diferente)
RAW = os.path.join("..", "01_data", "raw")
INTERIM = os.path.join("..", "01_data", "interim")
PROCESSED = os.path.join("..", "01_data", "processed")
OUT_TABLES = os.path.join("..", "04_outputs", "tables")

os.makedirs(PROCESSED, exist_ok=True)
os.makedirs(OUT_TABLES, exist_ok=True)

baseline_path = os.path.join(INTERIM, "baseline_table_ALSFRS_R.csv")
fvc_path = os.path.join(RAW, "PROACT_FVC.csv")
svc_path = os.path.join(RAW, "PROACT_SVC.csv")            # opcional (não entra na V1)
vitals_path = os.path.join(RAW, "PROACT_VITALSIGNS.csv")

base = pd.read_csv(baseline_path)
fvc = pd.read_csv(fvc_path)
vitals = pd.read_csv(vitals_path)

print("Baseline:", base.shape)
print("FVC:", fvc.shape)
print("VITALSIGNS:", vitals.shape)

base.head()


Baseline: (5436, 18)
FVC: (49110, 10)
VITALSIGNS: (84721, 36)


,subject_id,t0_delta_days,ALSFRS_R_t0,Age,Sex,Ethnicity,Race_Caucasian,Race_Black_African_American,Race_Asian,Race_Americ_Indian_Alaska_Native,Race_Hawaiian_Pacific_Islander,Race_Other,Race_Unknown,R_1_Dyspnea,R_2_Orthopnea,R_3_Respiratory_Insufficiency,Mode_of_Administration,ALSFRS_Responded_By
0,666,0.0,33.0,42.0,Male,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,2.0,3.0,4.0,NaN,NaN
1,935,0.0,35.0,45.0,Female,Non-Hispanic or Latino,1.0,NaN,NaN,NaN,NaN,NaN,NaN,3.0,4.0,4.0,NaN,NaN
2,1009,0.0,42.0,51.0,Male,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,3.0,4.0,4.0,NaN,NaN
3,1036,0.0,47.0,67.0,Female,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,4.0,4.0,NaN,NaN
4,1110,0.0,34.0,NaN,Male,Non-Hispanic or Latino,NaN,NaN,1.0,NaN,NaN,NaN,NaN,4.0,2.0,4.0,NaN,NaN


## 1) Funções utilitárias e normalização para sistema métrico

Nesta secção definimos funções para:
- converter unidades para o sistema métrico (ex.: kg, cm)
- selecionar a **última medição pré-t0** por doente (regra temporal)
- garantir consistência de colunas/nomes ao longo do pipeline

<b>Porque isto importa:</b> unidades inconsistentes e seleção temporal errada são duas fontes clássicas de ruído e leakage. Aqui garantimos que cada feature é “clinicamente interpretável” e temporalmente válida.


In [2]:
def to_kg(weight, unit):
    if pd.isna(weight): 
        return np.nan
    u = str(unit).strip().lower() if not pd.isna(unit) else ""
    if u in ["kg", "kgs", "kilogram", "kilograms"]:
        return float(weight)
    if u in ["lb", "lbs", "pound", "pounds"]:
        return float(weight) * 0.45359237
    return float(weight)  # fallback (se unidade desconhecida)

def to_cm(height, unit):
    if pd.isna(height): 
        return np.nan
    u = str(unit).strip().lower() if not pd.isna(unit) else ""
    if u in ["cm", "centimeter", "centimeters"]:
        return float(height)
    if u in ["m", "meter", "meters"]:
        return float(height) * 100.0
    if u in ["in", "inch", "inches"]:
        return float(height) * 2.54
    if u in ["ft", "feet"]:
        return float(height) * 30.48
    return float(height)  # fallback

def last_prebaseline(df, delta_col, base_df, t0_col="t0_delta_days"):
    """Mantém apenas registos com delta <= t0 e devolve o último (mais próximo de t0) por subject."""
    tmp = df.copy()
    tmp[delta_col] = pd.to_numeric(tmp[delta_col], errors="coerce")
    tmp = tmp.merge(base_df[["subject_id", t0_col]], on="subject_id", how="inner")
    tmp[t0_col] = pd.to_numeric(tmp[t0_col], errors="coerce")
    tmp = tmp.dropna(subset=[delta_col, t0_col])

    tmp = tmp[tmp[delta_col] <= tmp[t0_col]].copy()
    tmp = tmp.sort_values(["subject_id", delta_col], ascending=[True, True])
    # último registo por subject
    return tmp.groupby("subject_id", as_index=False).tail(1).copy()


## 2) Construir bloco: Vitals @ t0 (última medição pré-baseline)

Processo:
1) filtrar registos vitals para delta ≤ t0 (pre-baseline)
2) selecionar a última medição por doente
3) converter para unidades métricas (peso/altura) e derivar BMI (se aplicável)
4) renomear colunas com sufixo `_t0` para evitar ambiguidade

<b>Output:</b> `features_vitals_t0.csv` (1 linha por doente)


In [3]:
v_last = last_prebaseline(vitals, "Vital_Signs_Delta", base)

# conversões métricas
v_last["Weight_kg_t0"] = [to_kg(w,u) for w,u in zip(v_last.get("Weight"), v_last.get("Weight_Units"))]
v_last["Height_cm_t0"] = [to_cm(h,u) for h,u in zip(v_last.get("Height"), v_last.get("Height_Units"))]
v_last["BMI_t0"] = v_last["Weight_kg_t0"] / ((v_last["Height_cm_t0"]/100.0)**2)

# Seleção de colunas (só as que existirem no CSV)
candidate_cols = [
    "subject_id", "Vital_Signs_Delta",
    "Weight_kg_t0", "Height_cm_t0", "BMI_t0",
    "Pulse", "Respiratory_Rate", "Temperature",
    "Blood_Pressure_Systolic", "Blood_Pressure_Diastolic",
    "Baseline_Standing_BP_Systolic", "Baseline_Standing_BP_Diastolic",
    "Baseline_Supine_BP_Systolic", "Baseline_Supine_BP_Diastolic"
]
cols = [c for c in candidate_cols if c in v_last.columns]
vitals_features = v_last[cols].copy().rename(columns={"Vital_Signs_Delta":"vitals_delta_days"})

out_vitals = os.path.join(PROCESSED, "features_vitals_t0.csv")
vitals_features.to_csv(out_vitals, index=False)

print("Guardado:", out_vitals, "| shape:", vitals_features.shape)
vitals_features.head()


Guardado: ..\01_data\processed\features_vitals_t0.csv | shape: (4438, 14)


,subject_id,vitals_delta_days,Weight_kg_t0,Height_cm_t0,BMI_t0,Pulse,Respiratory_Rate,Temperature,Blood_Pressure_Systolic,Blood_Pressure_Diastolic,Baseline_Standing_BP_Systolic,Baseline_Standing_BP_Diastolic,Baseline_Supine_BP_Systolic,Baseline_Supine_BP_Diastolic
1,666,0.0,98.0,NaN,NaN,76.0,20.0,37.0,116.0,98.0,NaN,NaN,NaN,NaN
10,1009,0.0,78.9,184.0,23.304584,84.0,20.0,36.8,150.0,70.0,NaN,NaN,NaN,NaN
20,1110,0.0,57.0,160.0,22.265625,60.0,18.0,36.6,120.0,60.0,NaN,NaN,NaN,NaN
32,1137,0.0,80.1,162.6,30.296428,95.0,16.0,36.8,122.0,81.0,NaN,NaN,NaN,NaN
39,1333,12.0,64.1,NaN,NaN,88.0,20.0,37.1,130.0,84.0,NaN,NaN,NaN,NaN


## 3) Construir bloco: Função respiratória (FVC) @ t0

Processo:
1) filtrar FVC para delta ≤ t0
2) selecionar a última medição por doente
3) converter trials para numérico
4) consolidar num valor por doente (ex.: melhor tentativa = máximo das trials)
5) renomear colunas com sufixo `_t0`

<b>Nota:</b> o FVC é clinicamente relevante, mas costuma ter coverage inferior aos vitals; por isso este bloco é acompanhado por tabela de coverage.


In [4]:
f_last = last_prebaseline(fvc, "Forced_Vital_Capacity_Delta", base)

# converter trials para numérico
lit_cols = [c for c in ["Subject_Liters_Trial_1","Subject_Liters_Trial_2","Subject_Liters_Trial_3"] if c in f_last.columns]
pct_cols = [c for c in ["pct_of_Normal_Trial_1","pct_of_Normal_Trial_2","pct_of_Normal_Trial_3"] if c in f_last.columns]

for c in lit_cols + pct_cols:
    f_last[c] = pd.to_numeric(f_last[c], errors="coerce")

# best trial (máximo)
f_last["FVC_Liters_best_t0"] = f_last[lit_cols].max(axis=1) if lit_cols else np.nan
f_last["FVC_pctNormal_best_t0"] = f_last[pct_cols].max(axis=1) if pct_cols else np.nan

candidate_cols = [
    "subject_id", "Forced_Vital_Capacity_Delta",
    "FVC_Liters_best_t0", "FVC_pctNormal_best_t0",
    "subject_normal"
]
cols = [c for c in candidate_cols if c in f_last.columns]
fvc_features = f_last[cols].copy().rename(columns={"Forced_Vital_Capacity_Delta":"fvc_delta_days"})

out_fvc = os.path.join(PROCESSED, "features_fvc_t0.csv")
fvc_features.to_csv(out_fvc, index=False)

print("Guardado:", out_fvc, "| shape:", fvc_features.shape)
fvc_features.head()


Guardado: ..\01_data\processed\features_fvc_t0.csv | shape: (2896, 5)


,subject_id,fvc_delta_days,FVC_Liters_best_t0,FVC_pctNormal_best_t0,subject_normal
0,666,0.0,4.07,80.0,NaN
2,1036,0.0,NaN,129.0,NaN
7,1333,12.0,2.45,79.0,3.09
9,1492,0.0,3.67,77.0,4.74
16,2722,14.0,2.78,86.0,3.22


## 4) Construir tabela final de features (baseline + vitals + FVC)

Aqui fazemos o merge “horizontal” (1 linha por doente):
- base (demografia/ALSFRS baseline e colunas de identificação)
- +vitals_t0
- +fvc_t0

<b>Porque manter blocos separados antes do merge:</b>
- permite documentar progresso e decisões
- facilita ablation study (baseline-only vs +vitals vs +FVC)
- torna mais fácil identificar problemas de coverage por bloco


In [5]:
features_baseline_v1 = (
    base
    .merge(vitals_features, on="subject_id", how="left")
    .merge(fvc_features, on="subject_id", how="left")
)

out_final = os.path.join(PROCESSED, "features_baseline_v1.csv")
features_baseline_v1.to_csv(out_final, index=False)

print("Guardado:", out_final, "| shape:", features_baseline_v1.shape)
features_baseline_v1.head()


Guardado: ..\01_data\processed\features_baseline_v1.csv | shape: (5436, 35)


,subject_id,t0_delta_days,ALSFRS_R_t0,Age,Sex,Ethnicity,Race_Caucasian,Race_Black_African_American,Race_Asian,Race_Americ_Indian_Alaska_Native,...,Blood_Pressure_Systolic,Blood_Pressure_Diastolic,Baseline_Standing_BP_Systolic,Baseline_Standing_BP_Diastolic,Baseline_Supine_BP_Systolic,Baseline_Supine_BP_Diastolic,fvc_delta_days,FVC_Liters_best_t0,FVC_pctNormal_best_t0,subject_normal
0,666,0.0,33.0,42.0,Male,NaN,1.0,NaN,NaN,NaN,...,116.0,98.0,NaN,NaN,NaN,NaN,0.0,4.07,80.0,NaN
1,935,0.0,35.0,45.0,Female,Non-Hispanic or Latino,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1009,0.0,42.0,51.0,Male,NaN,NaN,1.0,NaN,NaN,...,150.0,70.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1036,0.0,47.0,67.0,Female,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,129.0,NaN
4,1110,0.0,34.0,NaN,Male,Non-Hispanic or Latino,NaN,NaN,1.0,NaN,...,120.0,60.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5) Guardar tabela de coverage (elegibilidade de features)

Criamos uma tabela de coverage por bloco:
- nº de doentes no baseline
- nº e % com pelo menos 1 feature disponível em Vitals
- nº e % com pelo menos 1 feature disponível em FVC
- nº e % com dados após merge final

<b>Uso na tese:</b>
- justificar por que certos blocos entram/saem do modelo
- explicar diferenças entre 3m e 6m (tamanho efetivo)
- suportar “Threats to Validity” (missingness/selection)


In [6]:
N_base = base["subject_id"].nunique()

cov_tbl = pd.DataFrame({
    "feature_block": ["VITALSIGNS", "FVC", "FINAL (baseline+vitals+fvc)"],
    "subjects_with_any_data": [
        vitals_features["subject_id"].nunique(),
        fvc_features["subject_id"].nunique(),
        features_baseline_v1["subject_id"].nunique() 
    ]
})
cov_tbl["coverage_pct_of_baseline"] = cov_tbl["subjects_with_any_data"] / N_base * 100

out_cov = os.path.join(OUT_TABLES, "step4_featureblock_coverage.csv")
cov_tbl.to_csv(out_cov, index=False)

cov_tbl


,feature_block,subjects_with_any_data,coverage_pct_of_baseline
0,VITALSIGNS,4438,81.640912
1,FVC,2896,53.274467
2,FINAL (baseline+vitals+fvc),5436,100.000000
